# Auditron — Multi-Agent Fraud Detection via Reinforcement Learning

A single language model plays **all roles simultaneously** in a procurement auction:
- 4 supplier agents (Honest / Shrewd / Random / Dishonest)
- 1 auditor agent — watches for cheaters, never sees actual part quality
- 1 buyer agent — follows the auditor's advice

Trained with GRPO (Group Relative Policy Optimization) via Unsloth + TRL.

> **GPU required.** Recommended: A100 or H100. T4 will work for short runs (20-40 steps).

In [ ]:
# Install dependencies
!pip install -q unsloth trl openenv-core
!pip install -q --upgrade transformers accelerate

In [ ]:
# Clone the repo
!git clone https://github.com/sfgeekgit/OpenEnvNorthflankEnv.git
%cd OpenEnvNorthflankEnv

In [ ]:
# Start the Auditron environment server (runs in background)
import subprocess, time
proc = subprocess.Popen(["python", "auditron_env/server.py"])
time.sleep(3)
print("Environment server started")

In [ ]:
import os

# Configuration — keep small for Colab demo
os.environ["MODEL_NAME"]           = "unsloth/Qwen2.5-1.5B-Instruct"
os.environ["NUM_TRAINING_STEPS"]   = "40"
os.environ["NUM_PROMPT_EPISODES"]  = "3"
os.environ["EVAL_EPISODES"]        = "1"

# Run training
exec(open("auditron_env/train.py").read())

In [ ]:
# View results from the final 50-round evaluation episode
import json, glob

ep_file = sorted(glob.glob("episodes_*.jsonl"))[-1]
print(f"Reading: {ep_file}")

final_summary = None
final_rounds = []
with open(ep_file) as f:
    for line in f:
        d = json.loads(line)
        if d.get("type") == "final_summary":
            final_summary = d
        elif d.get("type") == "final_round":
            final_rounds.append(d)

if final_summary:
    print("\n=== Episode Summary ===")
    print(f"Personalities: {final_summary['personalities']}")
    print(f"Total spend:   ${final_summary['total_spend']:,.0f}")
    print(f"Failures:      {final_summary['total_failures']}/50")
    print(f"Auditor TPR:   {final_summary.get('auditor_tpr') or 'N/A'}")
    print(f"Auditor FPR:   {final_summary.get('auditor_fpr') or 'N/A'}")
    print("\nSupplier profits:")
    for sid, profit in sorted(final_summary['supplier_profits'].items(),
                               key=lambda x: -x[1]):
        personality = final_summary['personalities'].get(sid, '?')
        print(f"  {sid} ({personality}): ${profit:,.0f}")

if final_rounds:
    print("\n=== Sample Auditor Reasoning ===")
    for r in final_rounds[::10]:  # every 10th round
        print(f"Round {r['round']}: pick={r['auditor_pick']} flags={r['auditor_flags']}")
        print(f"  Reason: {r['auditor_reason'][:200]}")

## What to look for

- **Supplier profits**: Does the Honest supplier rank higher after training?
- **Auditor flags**: Does the auditor learn to flag Dishonest/Shrewd suppliers?
- **Auditor reasoning**: Does the reasoning become more evidence-based over time?

For full training runs (150+ steps on H100), see the [training reports](https://github.com/sfgeekgit/OpenEnvNorthflankEnv).